# Stage 8B-2 — Existing Dataset Mean/Variance Cross-Validation

Bu testte yeni MATLAB Monte-Carlo reference üretmiyoruz.

Mevcut:

```text
variance_training_dataset_corrected_mu_rho_cf_v3.csv
```

içindeki **aynı environment + aynı W + aynı z** satırını Python/CUDA'da yeniden
simüle ediyoruz.

Karşılaştırma:

\[
\operatorname{mean}(Y)_{\rm Python}
\quad\leftrightarrow\quad
\texttt{meanEmp}_{\rm dataset}
\]

\[
\operatorname{var}(Y)_{\rm Python}
\quad\leftrightarrow\quad
\texttt{varEmp}_{\rm dataset}
\]

Dataset'teki `q05`, raw `prctile(Y,5)` değildir. Dataset'te `q05 == q05GammaFit`
ve Gamma q05, analitik `muSNR` + empirical `varEmp` ile hesaplanmıştır.

Bu yüzden ayrıca:

\[
q_{05,\Gamma}(\mu_{\rm SNR},\operatorname{var}_{Python})
\]

ile dataset `q05` değerini karşılaştırıyoruz.

Python'ın gerçek direct empirical:

\[
\operatorname{quantile}_{0.05}(Y)
\]

değerini de ayrıca raporluyoruz; bu ileride Direct-q05 NN için kullanılacak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, shutil, time
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required = [
    'ris_gpu_physics_stage1.py',
    'ris_gpu_channel_realizations_stage8b1.py',
    'ris_gpu_channel_native_stage8b2.py',
    'ris_gpu_precoder_stage5.py',
    'ris_gpu_ris_response_stage4.py',
    'stage8b2_dataset_mean_var_validation.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing=[
    f for f in required
    if not (ROOT/f).exists() and not (Path('/content')/f).exists()
]
assert not missing, "Eksik:\n"+"\n".join(missing)

from stage8b2_dataset_mean_var_validation import (
    load_validation_dataset,
    select_representative_rows,
    validate_dataset_row,
    summarize_results,
)

device='cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

In [ ]:
# Dataset'i Drive'dan local SSD'ye kopyala.
DRIVE_CSV = ROOT / 'variance_training_dataset_corrected_mu_rho_cf_v3.csv'
assert DRIVE_CSV.exists(), DRIVE_CSV

LOCAL_CSV = Path('/content/variance_training_dataset_corrected_mu_rho_cf_v3.csv')

if (
    not LOCAL_CSV.exists()
    or LOCAL_CSV.stat().st_size != DRIVE_CSV.stat().st_size
):
    print("CSV local SSD'ye kopyalanıyor...")
    shutil.copy2(DRIVE_CSV,LOCAL_CSV)

print("CSV:",LOCAL_CSV)
print("GiB:",LOCAL_CSV.stat().st_size/1024**3)

D=load_validation_dataset(str(LOCAL_CSV))
print("Rows:",len(D))
print("nRIS:",sorted(D.nRIS.unique()))
print("nEval:",sorted(D.nEval.unique()))

## 8 representative rows

Test-interpolation splitinden 8 satır:

- 2 × nRIS=64
- 2 × nRIS=128
- 2 × nRIS=256
- 2 × nRIS=512

LOS/NLOS ve mixed LOS-NLOS branch'leri de kapsanıyor.

In [ ]:
S=select_representative_rows(
    D,
    split='test_interpolation',
)

display(
    S[
        [
            'bankID','pairID','scenario_BR','scenario_RU',
            'nT1','nT2','nR1','nR2',
            'nRIS','WIdx_i11','WIdx_i12','WIdx_i2',
            'meanEmp','varEmp','q05'
        ]
    ]
)

## N = 100,000 independent CUDA Monte Carlo

Burada MATLAB RNG ile sample-by-sample eşleşme aramıyoruz.

Aynı fiziksel distribution'dan bağımsız:

\[
N_{\rm Python}=100000
\]

örnek üretiyoruz.

Dataset empirical istatistikleri ise `nEval=9997` sample'dan gelmektedir.

In [ ]:
N_PYTHON=100_000
CHUNK=1024

rows=[]

if torch.cuda.is_available():
    torch.cuda.synchronize()

T0=time.perf_counter()

for i,row in S.iterrows():

    print(
        f"\n[{i+1}/{len(S)}] "
        f"bank={int(row.bankID)} pair={int(row.pairID)} | "
        f"{row.scenario_BR} / {row.scenario_RU} | "
        f"nRIS={int(row.nRIS)}"
    )

    r=validate_dataset_row(
        row,
        N_python=N_PYTHON,
        chunk_size=CHUNK,
        device=device,
        parity=False,
        store_y=True,
    )

    rows.append(r)

    print(
        f"  mean : dataset={r['meanEmp_dataset']:.6g} "
        f"python={r['meanPython']:.6g} "
        f"err={100*r['meanRelErr_vs_dataset']:.3f}%"
    )
    print(
        f"  var  : dataset={r['varEmp_dataset']:.6g} "
        f"python={r['varPython']:.6g} "
        f"err={100*r['varRelErr_vs_dataset']:.3f}%"
    )
    print(
        f"  Gamma q05: dataset={r['q05Gamma_dataset']:.6g} "
        f"python-var={r['q05Gamma_from_pythonVar']:.6g} "
        f"err={100*r['q05GammaRelErr']:.3f}%"
    )
    print(
        f"  direct empirical q05 Python={r['q05EmpPython_direct']:.6g}"
    )
    print(
        f"  time={r['seconds']:.3f}s | "
        f"{r['realization_pairs_per_second']:,.0f} pair/s"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

TOTAL_WALL=time.perf_counter()-T0

R=pd.DataFrame(rows)

print("\nTOTAL WALL:",f"{TOTAL_WALL:.3f} s = {TOTAL_WALL/60:.3f} min")

In [ ]:
cols=[
    'bankID','pairID','scenario_BR','scenario_RU','nRIS',
    'meanEmp_dataset','meanPython','meanRelErr_vs_dataset',
    'muSNR_analytic','meanRelErr_vs_muSNR',
    'varEmp_dataset','varPython','varRelErr_vs_dataset',
    'q05Gamma_dataset','q05Gamma_from_pythonVar','q05GammaRelErr',
    'q05EmpPython_direct','gammaVsDirectPython_relGap',
    'seconds','realization_pairs_per_second',
]

display(R[cols])

In [ ]:
summary=summarize_results(R)

print("="*78)
print("STAGE 8B-2 — DATASET CROSS-VALIDATION SUMMARY")
print("="*78)

for name,label in [
    ('meanRelErr_vs_dataset','mean Python vs meanEmp'),
    ('meanRelErr_vs_muSNR','mean Python vs analytic muSNR'),
    ('varRelErr_vs_dataset','var Python vs varEmp'),
    ('q05GammaRelErr','Gamma q05 using Python var vs dataset q05'),
]:
    print(
        f"{label:<43s} | "
        f"median={100*summary[name+'_median']:.3f}% | "
        f"P90={100*summary[name+'_p90']:.3f}% | "
        f"max={100*summary[name+'_max']:.3f}%"
    )

print()
print(f"Kernel summed time : {summary['total_seconds']:.3f} s")
print(f"Notebook wall time : {TOTAL_WALL:.3f} s")
print(
    "Overall throughput:",
    f"{summary['overall_pairs_per_second']:,.0f} realization-pair/s"
)

### PASS'i nasıl yorumlayacağız?

Bu iki Monte-Carlo seti bağımsızdır:

- eski MATLAB dataset: `nEval = 9997`
- yeni Python: `N = 100000`

Bu nedenle sıfır hata beklemiyoruz.

Öncelikle şu iki metriğe bakacağız:

\[
\boxed{\text{mean error}}
\qquad
\boxed{\text{variance error}}
\]

ve downstream kontrol olarak:

\[
\boxed{q_{05,\Gamma}(\mu_{\rm SNR},var_{Python})}
\]

Dataset `q05` ile karşılaştırılacak.

Sonuçları gördükten sonra B2'yi PASS kapatıp doğrudan **8B3 N-convergence +
multi-W/multi-z empirical label engine** aşamasına geçeceğiz.

In [ ]:
OUT=Path('/content/stage8b2_dataset_cross_validation.csv')
R.to_csv(OUT,index=False)
print("Saved:",OUT)